<a href="https://colab.research.google.com/github/sbhmrj/AI-Customer-Complaint-Routing-Engine/blob/main/AI_Complaint_Routing_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers accelerate torch sentencepiece

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    device_map="cpu"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
import json

def route_complaint(complaint):

    prompt = f"""
You are an AI Complaint Routing Engine.

Return ONLY JSON.

Example:

{{
"category":"Billing",
"priority":"High",
"summary":"Double payment deduction",
"assigned_team":"Finance Team",
"sla":"4 Hours"
}}

Complaint:
{complaint}
"""

    inputs = tokenizer(prompt, return_tensors="pt")

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.2
    )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return response

In [ ]:
complaint = """
Amount deducted twice while paying electricity bill.
"""

print(route_complaint(complaint))


You are an AI Complaint Routing Engine.

Return ONLY JSON.

Example:

{
"category":"Billing",
"priority":"High",
"summary":"Double payment deduction",
"assigned_team":"Finance Team",
"sla":"4 Hours"
}

Complaint:

Amount deducted twice while paying electricity bill.

The user is a resident of a small town and the complaint is about a duplicate deduction.

The user is a resident of a small town and the complaint is about a duplicate deduction.

The user is a resident of a small town and the complaint is about a duplicate deduction.

The user is a resident of a small town and the complaint is about a duplicate deduction.

The user is a resident of a small town and the complaint is about a duplicate deduction.

The user is a resident of a small town and the complaint is about a duplicate deduction.

The user is a resident of a small town and the complaint is about a duplicate deduction.

The user is a resident of a small town and the complaint is about a duplicate deduction.

The user is

In [ ]:
def route_complaint(complaint):

    messages = [
        {
            "role": "system",
            "content": """You are an AI Complaint Routing Engine.

Return ONLY valid JSON.

Format:
{
 "category":"",
 "priority":"",
 "summary":"",
 "assigned_team":"",
 "sla":""
}
"""
        },
        {
            "role": "user",
            "content": complaint
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt")

    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    return response

In [ ]:
complaint = """
Amount deducted twice while paying electricity bill.
"""

print(route_complaint(complaint))

<think>
Okay, the user mentioned that the amount was deducted twice while paying their electricity bill. Let me think about how to structure the response.

First, the category. Since it's about a payment issue, maybe "payment issue" or "billing error". The priority might be high because it's a serious problem. The summary should explain the issue clearly. The assigned team could be the billing department. The SLA (Service Level Agreement) might be set to handle such cases. I need to


In [ ]:
data = [
    ("Amount deducted twice while paying bill", "Billing"),
    ("UPI transaction failed", "UPI"),
    ("Credit card charged incorrectly", "Card"),
    ("Loan EMI deducted twice", "Loan"),
    ("Fraud transaction on account", "Fraud"),
    ("KYC update issue", "KYC"),
]

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

X = [
    "Amount deducted twice while paying bill",
    "UPI transaction failed",
    "Credit card charged incorrectly",
    "Loan EMI deducted twice",
    "Fraud transaction on account",
    "KYC update issue"
]

y = [
    "Billing",
    "UPI",
    "Card",
    "Loan",
    "Fraud",
    "KYC"
]

model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", MultinomialNB())
])

model.fit(X, y)

Pipeline(steps=[('tfidf', TfidfVectorizer()), ('clf', MultinomialNB())])

In [ ]:
team_mapping = {
    "Billing": "Finance Team",
    "UPI": "Digital Banking Team",
    "Card": "Card Operations Team",
    "Loan": "Loan Processing Team",
    "Fraud": "Fraud Investigation Team",
    "KYC": "Customer Service Team"
}

In [ ]:
def get_priority(text):

    text = text.lower()

    if "fraud" in text:
        return "Critical"

    if "deducted twice" in text:
        return "High"

    if "failed" in text:
        return "Medium"

    return "Low"

In [ ]:
sla_mapping = {
    "Critical": "1 Hour",
    "High": "4 Hours",
    "Medium": "24 Hours",
    "Low": "48 Hours"
}

In [ ]:
def route_complaint(complaint):

    category = model.predict([complaint])[0]

    priority = get_priority(complaint)

    return {
        "category": category,
        "priority": priority,
        "summary": complaint[:80],
        "assigned_team": team_mapping[category],
        "sla": sla_mapping[priority]
    }

In [ ]:
complaint = "Amount deducted twice while paying electricity bill"

result = route_complaint(complaint)

print(result)

{'category': np.str_('Billing'), 'priority': 'High', 'summary': 'Amount deducted twice while paying electricity bill', 'assigned_team': 'Finance Team', 'sla': '4 Hours'}


In [ ]:
import json

complaint = "Amount deducted twice while paying electricity bill"

result = route_complaint(complaint)

print(json.dumps(result, indent=4))

{
    "category": "Billing",
    "priority": "High",
    "summary": "Amount deducted twice while paying electricity bill",
    "assigned_team": "Finance Team",
    "sla": "4 Hours"
}


In [ ]:
import gradio as gr
import json

def predict(complaint):
    return json.dumps(route_complaint(complaint), indent=4)

demo = gr.Interface(
    fn=predict,
    inputs="textbox",
    outputs="textbox",
    title="AI Complaint Routing Engine"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0dbb92d599b3321d79.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB


# ==========================
# Training Data
# ==========================

X = [
    "Amount deducted twice while paying bill",
    "UPI transaction failed",
    "Credit card charged incorrectly",
    "Loan EMI deducted twice",
    "Fraud transaction on account",
    "KYC update issue",
    "Payment debited but merchant not received",
    "Unauthorized card transaction",
    "Internet banking login problem",
    "ATM cash not dispensed but account debited",
]

y = [
    "Billing",
    "UPI",
    "Card",
    "Loan",
    "Fraud",
    "KYC",
    "UPI",
    "Fraud",
    "Digital Banking",
    "ATM"
]


# ==========================
# Model Training
# ==========================

model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", MultinomialNB())
])

model.fit(X, y)


# ==========================
# Team Mapping
# ==========================

team_mapping = {
    "Billing": "Finance Team",
    "UPI": "Digital Banking Team",
    "Card": "Card Operations Team",
    "Loan": "Loan Processing Team",
    "Fraud": "Fraud Investigation Team",
    "KYC": "Customer Service Team",
    "Digital Banking": "Digital Banking Team",
    "ATM": "ATM Operations Team"
}


# ==========================
# Priority Logic
# ==========================

def get_priority(text):

    text = text.lower()

    critical_words = [
        "fraud",
        "unauthorized",
        "scam",
        "stolen"
    ]

    high_words = [
        "debited",
        "deducted",
        "failed",
        "charged",
        "payment not received",
        "bill unpaid",
        "merchant not received"
    ]

    if any(word in text for word in critical_words):
        return "Critical"

    if any(word in text for word in high_words):
        return "High"

    return "Medium"


# ==========================
# SLA Logic
# ==========================

sla_mapping = {
    "Critical": "1 Hour",
    "High": "4 Hours",
    "Medium": "24 Hours",
    "Low": "48 Hours"
}


# ==========================
# Main Routing Function
# ==========================

def route_complaint(complaint):

    category = model.predict([complaint])[0]

    priority = get_priority(complaint)

    team = team_mapping.get(
        category,
        "Customer Service Team"
    )

    sla = sla_mapping.get(
        priority,
        "24 Hours"
    )

    confidence = round(
        max(model.predict_proba([complaint])[0]) * 100,
        2
    )

    return {
        "category": category,
        "priority": priority,
        "confidence": f"{confidence}%",
        "summary": complaint,
        "assigned_team": team,
        "sla": sla
    }


# ==========================
# Gradio UI
# ==========================

demo = gr.Interface(
    fn=route_complaint,
    inputs=gr.Textbox(
        lines=6,
        placeholder="Enter customer complaint here...",
        label="Complaint"
    ),
    outputs=gr.JSON(label="Routing Result"),
    title="AI Complaint Routing Engine",
    description="Automatically classify complaints, assign teams, and predict SLA."
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://18266a296f7b1f1389.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
